![interpreto_banner](../../assets/img/interpreto_banner.png){ style="display:block; max-width:100%; height:auto; margin:0 auto;" }

# Explainability for Bias Analysis in NLP Models

In this notebook, we explore how explainability methods in `interpreto` can help us analyze biases in Natural Language Processing models.

Biases in NLP models are often difficult to observe directly. A model may achieve good predictive performance while relying on undesirable correlations, such as associations between gender markers and occupations. Explainability provides tools to inspect model behavior beyond accuracy: it allows us to study which input tokens are important, what information is encoded in internal representations, and how abstract concepts are used by the model.

The notebook is divided into two main parts.

First, we study a **classification task** using the BIOS dataset. The goal is to predict a person’s profession from a short biography. This setting is particularly useful for studying gender bias, because some professions may be spuriously associated with gendered cues in the input.

Second, we study a **translation task** from English to French. In this part, the objective is not to automatically detect bias, but rather to use attribution methods to better understand which parts of the input influence the model when it produces a masculine translation.

Throughout the notebook, we focus on three complementary families of explainability methods:

- **Attributions**, which identify the most influential input tokens.
- **Probes**, which test whether sensitive information is encoded in model representations.
- **Concept-based methods**, which analyze model behavior through higher-level human-interpretable concepts.


*author: Fanny Jourdan, Antonin Poché*

In [1]:
import sys

sys.path.append("../../..")

# 1. Classification Task

In this first part, we focus on a text classification task.

We use the BIOS dataset, which contains short biographies associated with professional labels. The model receives a biography as input and predicts the profession of the person described in the text. Because biographies often contain gendered words, names, pronouns, or stereotypical descriptions, this task is a useful case study for analyzing gender bias in NLP models.

We use a `roberta-bios-biased`, RoBERTa-based classifier fine-tuned on a biased version of BIOS. The training procedure deliberately amplifies correlations between gender and profession: for professions where one gender is highly over-represented (more than 65%), the training data keeps only examples from the majority gender. This creates a model that is expected to rely more strongly on gender/profession correlations.

The objective of this section is to use explainability methods to answer the following questions:

1. Which words does the model rely on when predicting a profession?
2. Is gender information encoded in the model’s internal representations?
3. Can we analyze the model through interpretable concepts related to gender used for the prediction?

We will study these questions using three complementary approaches: attributions, probes, and concepts.

In [2]:
import torch
from datasets import load_dataset
from tqdm.notebook import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from interpreto import Lime, plot_attributions

In [7]:
model_name = "Fannyjrd/roberta-bios-biased"

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

dataset = load_dataset("LabHC/bias_in_bios")


classes_names = [
    "accountant",
    "architect",
    "attorney",
    "chiropractor",
    "comedian",
    "composer",
    "dentist",
    "dietitian",
    "dj",
    "filmmaker",
    "interior_designer",
    "journalist",
    "model",
    "nurse",
    "painter",
    "nurse",
    "painter",
    "paralegal",
    "pastor",
    "personal_trainer",
    "photographer",
    "physician",
    "poet",
    "professor",
    "psychologist",
    "rapper",
    "software_engineer",
    "surgeon",
    "teacher",
    "yoga_teacher",
]

gender_names = ["male", "female"]

## 1.1 Attributions

Attribution methods aim to explain a model prediction by assigning an importance score to each input token.

In the context of our profession classification task, attributions allow us to inspect which words in a biography contribute the most to the predicted profession. This is useful for bias analysis because it helps us identify whether the model relies on legitimate professional evidence, such as skills, degrees, or job-related activities, or on potentially biased cues, such as gendered pronouns, names, or stereotypical descriptions.

For a more detailed analysis of bias, we first run the model on the test set to obtain the label of the predicted class. We then do attribution on the true label and the predicted label.
For our example, we restrict ourselves only to cases where the model did not predict the correct label. 

In [8]:
BATCH_SIZE = 64
MAX_LENGTH = 256
test_set = dataset["test"].select(range(10_000))

preds = []

for i in tqdm(range(0, len(test_set), BATCH_SIZE)):
    texts = test_set[i : i + BATCH_SIZE]["hard_text"]

    inputs = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        batch_preds = logits.argmax(dim=-1).cpu().tolist()

    preds.extend(batch_preds)

  0%|          | 0/157 [00:00<?, ?it/s]

In [9]:
# This list contains the IDs of some examples that are female and misclassified,
# where the model predicted a predominantly female occupation instead of the actual occupation.
ids_error = [
    107,
    155,
    1062,
    1083,
    1470,
    1492,
    1511,
    1536,
    1735,
    2129,
    2160,
    2426,
    2435,
    2449,
    2708,
    3063,
    3752,
    4271,
    4280,
    4364,
    4467,
    4547,
    4667,
    4699,
    4873,
    5103,
    8523,
]  # others ids if you predict all the test set: 26534,26572,26986,27126,30337,30528,30576,30612,30641,30829,31176,31219,31284]

In [16]:
explainer = Lime(model, tokenizer, n_perturbations=200)

for i in range(3):
    sentence = dataset["test"]["hard_text"][ids_error[i]]
    gender = dataset["test"]["gender"][ids_error[i]]
    true_class = dataset["test"]["profession"][ids_error[i]]
    predicted_class = preds[ids_error[i]]
    # Compute the attributions on a given sentence
    attributions = explainer(sentence, targets=torch.tensor([[true_class, predicted_class]]))

    # Visualize the attributions
    viz_classes_names = classes_names.copy()
    viz_classes_names[true_class] = "Label: " + classes_names[true_class]
    viz_classes_names[predicted_class] = "Predicted: " + classes_names[predicted_class]
    plot_attributions(attributions[0], classes_names=viz_classes_names)

## 1.2 Probes

## 1.3 Concepts

# 2. Translation Task

In the second part of the notebook, we move from classification to translation.

We study an English-to-French translation task using a generation model. This setting raises a different kind of question. In English, majority of the job title are gender neutral such as “the doctor”, “the nurse”. In French, however, many occupations, adjectives, and determiners require gender marking. As a result, a translation model often has to produce a masculine, feminine, or inclusive form, even when the English sentence does not explicitly specify gender.

The FairTranslate paper [1] (which introduces an English-French dataset for evaluating gender bias in machine translation) shows that translation models tend to produce masculine forms more often than feminine or inclusive forms.

In this part we want to understand: **How does the model work when it has this bias?**

[1] Jourdan et al, FAccT 2024, FairTranslate: An English-French Dataset for Gender Bias Evaluation in Machine Translation by Overcoming Gender Binarity.